In [2]:
import json
import os
import shutil

import pandas as pd

In [48]:
def load_dataset(input_annotations_file: str, input_img_dir: str, output_dir: str):
    with open(input_annotations_file, "r") as f:
        dataset = json.load(f)
    df_annotation = pd.DataFrame(dataset['annotations']).drop(['segmentation', 'attribute_ids'], axis='columns')
    df_images = pd.DataFrame(dataset['images'])
    df_categories = pd.DataFrame(dataset['categories'])

    df_dataset = pd.merge(df_images, df_annotation, how='inner', left_on='id', right_on='image_id')
    df_dataset = pd.merge(df_dataset, df_categories, how='inner', left_on='category_id', right_on='id')
    df_dataset['path_to_image'] = input_img_dir + df_dataset['file_name']
    return df_dataset


In [49]:
def save_labels_one_image(df_one_image: pd.DataFrame, output_folder: str, subfolder: str):
    path_to_image = df_one_image.iloc[0]['path_to_image']
    image_id = df_one_image.iloc[0]['image_id']
    image_width = df_one_image.iloc[0]['width']
    image_height = df_one_image.iloc[0]['height']

    def format_one_box(one_line) -> str:
        label_index = str(one_line['category_id'])

        [x_center, y_center, width, height] = one_line['bbox']
        x_center_normalised, y_center_normalised = (x_center+width/2) / image_width, (y_center+height/2) / image_height
        width_normalised, height_normalised = width / image_width, height / image_height
        box = ' '.join(map(str, [x_center_normalised, y_center_normalised, width_normalised, height_normalised]))

        line = f'{label_index} {box}'
        return line

    boxes_formatted = df_one_image.apply(format_one_box, axis='columns').tolist()

    with open(os.path.join(output_folder, 'labels', subfolder, f'{image_id}.txt'), 'w') as f:
        f.write('\n'.join(boxes_formatted))
    shutil.copy(path_to_image, os.path.join(output_folder, 'images', subfolder, f'{image_id}.jpg'))

    return image_id


In [50]:
path_to_json_validation = "./dataset/fashionpedia2020/instances_attributes_val2020.json"
path_to_image_folder_validation = './dataset/fashionpedia2020/val_test2020/test/'
output_folder_validation = 'val2020'
fashionpedia_dataset_validation = load_dataset(path_to_json_validation, path_to_image_folder_validation,
                                               'dataset/fashionpedia-yolo')
fashionpedia_dataset_validation.groupby(['image_id']).apply(save_labels_one_image, './dataset/fashionpedia-yolo',
                                                            output_folder_validation)

In [51]:
path_to_json_train = "./dataset/fashionpedia2020/instances_attributes_train2020.json"
path_to_image_folder_train = './dataset/fashionpedia2020/train2020/train/'
output_folder_train = 'train2020'
fashionpedia_dataset_train = load_dataset(path_to_json_train, path_to_image_folder_train, 'dataset/fashionpedia-yolo')
fashionpedia_dataset_train.groupby(['image_id']).apply(save_labels_one_image, './dataset/fashionpedia-yolo',
                                                       output_folder_train)

In [52]:
from matplotlib import pyplot as plt

COLORS = ['lightcoral', 'yellowgreen', 'darkturquoise', 'hotpink', 'mediumslateblue']


def plot_results(pil_img, boxes, class_labels):
    plt.figure(figsize=(16, 10))
    plt.imshow(pil_img)
    ax = plt.gca()
    colors = COLORS * 100
    for [xcenter, ycenter, xlength, ylength], c, label in zip(boxes, colors, class_labels):
        xmin = (xcenter-(xlength/2)) * pil_img.width
        xmax = (xcenter+(xlength/2)) * pil_img.width
        ymin = (ycenter-(ylength/2)) * pil_img.height
        ymax = (ycenter+(ylength/2)) * pil_img.height
        
        print(xmin, ymin)
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=2))
        ax.text(xmin, ymin, str(label), fontsize=15,
                bbox=dict(facecolor=c, alpha=0.5))
    plt.axis('off')
    plt.show()

In [53]:
from PIL import Image

pil_image = Image.open('dataset/fashionpedia-yolo/images/train2020/23.jpg')
with open('dataset/fashionpedia-yolo/labels/train2020/23.txt' ) as f:
    labels = f.readlines()

class_labels  = []
box_center_length = []
for a_label in labels:
    labels_split = a_label.split(' ')
    class_labels.append(int(labels_split[0]))
    box_center_length.append(list(map(float, labels_split[1:])))
plot_results(pil_image,box_center_length, class_labels)